In [1]:
import numpy as np
import librosa

def extract_stft(
    signal,
    sr=16000,
    n_fft=512,
    hop_length=160,   # 10 ms
    win_length=400    # 25 ms
):

    stft = librosa.stft(
        signal,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window="hann"
    )

    # magnitude spectrogram
    spec = np.abs(stft)

    # log scale
    log_spec = librosa.amplitude_to_db(spec)

    return log_spec.T   # shape: (frames, freq_bins)

In [2]:
import numpy as np
import librosa
from skimage.feature import local_binary_pattern


def build_stft_dataset(
    wav_list,
    sr=16000,
    segment_sec=2,
    out_path="X_stft.npy"
):

    all_feats = []
    segment_len = int(segment_sec * sr)

    for wav_path in wav_list:

        print("Processing:", wav_path)

        y, sr = librosa.load(wav_path, sr=sr)

        # nếu audio < 2s
        if len(y) < segment_len:
            segments = [y]
        else:
            num_segments = len(y) // segment_len
            segments = [
                y[i*segment_len:(i+1)*segment_len]
                for i in range(num_segments)
            ]

        for segment in segments:

            stft_feat = extract_stft(segment, sr)

            lbp = local_binary_pattern(
                stft_feat,
                P=8,
                R=1,
                method="default"
            )

            hist, _ = np.histogram(
                lbp.ravel(),
                bins=256,
                range=(0, 256)
            )

            all_feats.append(hist)

        print(f"{wav_path} -> {len(segments)} segments")

    X = np.vstack(all_feats)

    np.save(out_path, X)

    print("Saved:", out_path, X.shape)

    return X

In [4]:
import os
CLASS = "bonafide"
SET = "eval"

DIR = f"E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed"

wav_list = [
    os.path.join(DIR, f)
    for f in os.listdir(DIR)
    if f.endswith(".wav")
]
print(wav_list)
X_mfcc = build_stft_dataset(
    wav_list,
    out_path=f"E:/PythonFile/Project/Voice-Activity-Detect/data/feature/train/stft_lbp_speech_mix_aurora.npy"
)

print(len(wav_list))
print(X_mfcc.shape)

['E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed\\1.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed\\10.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed\\100.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed\\101.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed\\102.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed\\103.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed\\104.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed\\105.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed\\106.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed\\107.wav', 'E:/PythonFile/Project/Voice-Activity-Detect/data/processed/aurora/speech-mixed\\108.wav', '